# Tiny Shakespeare Fast-Iteration Research Notebook

Run local tiny-shakespeare baselines without SLURM, evaluate all models with the same Gen-PPL protocol, and iterate quickly with small ablation grids.

Output layout:
- `outputs/tiny_shakespeare/notebook_runs/<run_name>/...`
- `outputs/tiny_shakespeare/notebook_runs/<run_name>/eval/gen_ppl/<timestamp>/samples.pt`
- `outputs/tiny_shakespeare/notebook_runs/<run_name>/eval/gen_ppl/<timestamp>/samples.txt`
- `outputs/tiny_shakespeare/notebook_runs/<run_name>/eval/gen_ppl/<timestamp>/gen_ppl_metrics.json`
- `outputs/tiny_shakespeare/notebook_runs/summary/experiments.csv`


In [3]:
import os
import sys
import json
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def detect_repo_root(start: Path | None = None) -> Path:
    probe = (start or Path.cwd()).resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'discrete_diffusion').exists():
            return candidate
    raise RuntimeError(f'Could not detect repo root from {probe}')


REPO_ROOT = detect_repo_root()
os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / 'src')
existing_pythonpath = os.environ.get('PYTHONPATH', '')
if existing_pythonpath:
    if src_path not in existing_pythonpath.split(':'):
        os.environ['PYTHONPATH'] = f'{src_path}:{existing_pythonpath}'
else:
    os.environ['PYTHONPATH'] = src_path

os.environ['WANDB_MODE'] = 'disabled'

DATA_CACHE_DIR = Path('/lustre/groups/bauer/projects/discrete_diff_cdb/Discrete_Diffusion_RAE/datasets/')
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'tiny_shakespeare' / 'notebook_runs'
SUMMARY_DIR = OUTPUT_ROOT / 'summary'
SUMMARY_CSV = SUMMARY_DIR / 'experiments.csv'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

CUDA_AVAILABLE = torch.cuda.is_available()

print(f'repo_root={REPO_ROOT}')
print(f'python={sys.executable}')
print(f'PYTHONPATH={os.environ["PYTHONPATH"]}')
print(f'WANDB_MODE={os.environ["WANDB_MODE"]}')
print(f'torch={torch.__version__} cuda_available={CUDA_AVAILABLE}')
if not CUDA_AVAILABLE:
    print('No CUDA detected: continuous baseline will be auto-skipped; MDLM/JEPA will use CPU-safe overrides.')


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# !pip install pandas

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 46.8 MB/s  0:00:006m0:00:01
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [ ]:
PROFILE_NAME = 'FAST_LOCAL'

SEQ_LEN = 128
EVAL_MODEL = 'gpt2'
NUM_SAMPLES = 64

FAST_STEP_BUDGETS = {
    'mdlm': 1000,
    'latent_jepa': 1000,
    'continuous': 600,
}

COMMON_CADENCE = {
    'trainer.val_check_interval': 200,
    'callbacks.checkpoint_every_n_steps.every_n_train_steps': 200,
    'trainer.log_every_n_steps': 50,
    'eval.generate_samples': 'true',
    'eval.save_validation_samples': 'true',
    'sampling.steps': 16,
    'sampling.num_sample_batches': 1,
    'sampling.num_sample_log': 4,
}

RUN_MDLM = True
RUN_JEPA = True
RUN_CONTINUOUS = True
if not CUDA_AVAILABLE:
    RUN_CONTINUOUS = False

MODEL_ENABLED = {
    'mdlm': RUN_MDLM,
    'latent_jepa': RUN_JEPA,
    'continuous': RUN_CONTINUOUS,
}

EVAL_NUM_STEPS = {
    'mdlm': 128,
    'latent_jepa': 64,
    'continuous': 64,
}

EVAL_GEN_BATCH = {
    'mdlm': 16,
    'latent_jepa': 16,
    'continuous': 16,
}

EVAL_BATCH_SIZE = 4
MAX_LENGTH = SEQ_LEN
ABLATION_MAX_STEPS = 400

print(f'profile={PROFILE_NAME}')
print(f'seq_len={SEQ_LEN} eval_model={EVAL_MODEL} num_samples={NUM_SAMPLES}')
print(f'step_budgets={FAST_STEP_BUDGETS}')
print(f'model_enabled={MODEL_ENABLED}')


In [ ]:
RESULT_ROWS: list[dict] = []
VARIANT_OVERRIDES: dict[tuple[str, str], list[str]] = {}


# Use the same command pattern as existing training entrypoints.
PYTHON_BIN = 'python'


def run_cmd(cmd: list[str], env: dict | None = None) -> subprocess.CompletedProcess:
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    printable = ' '.join(shlex.quote(part) for part in cmd)
    print(f'\n$ {printable}')

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line)

    return_code = process.wait()
    output = ''.join(output_lines)
    completed = subprocess.CompletedProcess(cmd, return_code, output)

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd, output=output)

    return completed


def train_overrides(
    model_key: str,
    run_dir: str,
    run_name: str | None = None,
    max_steps: int | None = None,
    extra_overrides: list[str] | None = None,
) -> list[str]:
    if model_key not in {'mdlm', 'latent_jepa', 'continuous'}:
        raise ValueError(f'Unsupported model_key={model_key}')

    run_name = run_name or Path(run_dir).name
    max_steps = max_steps if max_steps is not None else FAST_STEP_BUDGETS[model_key]

    common = [
        'data=tiny_shakespeare',
        f'data.cache_dir={DATA_CACHE_DIR.as_posix()}',
        f'model.length={SEQ_LEN}',
        'loader.num_workers=4',
        'trainer.num_nodes=1',
        'trainer.devices=1',
        'checkpointing.resume_from_ckpt=false',
        'callbacks.checkpoint_every_n_steps.save_last=true',
        'callbacks.checkpoint_monitor.save_top_k=1',
        f"callbacks.checkpoint_every_n_steps.every_n_train_steps={COMMON_CADENCE['callbacks.checkpoint_every_n_steps.every_n_train_steps']}",
        f"trainer.val_check_interval={COMMON_CADENCE['trainer.val_check_interval']}",
        f"trainer.log_every_n_steps={COMMON_CADENCE['trainer.log_every_n_steps']}",
        f"eval.generate_samples={COMMON_CADENCE['eval.generate_samples']}",
        f"eval.save_validation_samples={COMMON_CADENCE['eval.save_validation_samples']}",
        f"sampling.steps={COMMON_CADENCE['sampling.steps']}",
        f"sampling.num_sample_batches={COMMON_CADENCE['sampling.num_sample_batches']}",
        f"sampling.num_sample_log={COMMON_CADENCE['sampling.num_sample_log']}",
        f'hydra.run.dir={run_dir}',
        f'checkpointing.save_dir={run_dir}',
        'wandb.project=tiny_shakespeare_notebook',
        f'wandb.name={run_name}',
        f'wandb.save_dir={run_dir}/wandb',
    ]

    if model_key == 'mdlm':
        model_specific = [
            'algo=mdlm',
            'model=small',
            'loader.global_batch_size=64',
            'loader.eval_global_batch_size=16',
            f'trainer.max_steps={max_steps}',
            'trainer.precision=bf16-mixed',
        ]
    elif model_key == 'latent_jepa':
        model_specific = [
            'algo=jepa',
            'model=latent_jepa_small',
            'algo.stage=1',
            'loader.global_batch_size=64',
            'loader.eval_global_batch_size=16',
            f'trainer.max_steps={max_steps}',
            'strategy=single-device',
            'trainer.precision=16-mixed',
            'callbacks.latent_eval.eval_frequency=50000',
        ]
    else:
        model_specific = [
            'algo=continuous_embedding_diffusion',
            'model=continuous_embedding',
            'noise=cosine',
            'sampling=continuous_embedding',
            'algo.stage=1',
            'algo.parameterization=x0',
            'algo.loss_type=simple',
            'algo.lambda_ce=0.1',
            'algo.embedding_provider=legacy_contextual',
            'model.encoder_name=Qwen/Qwen3-Embedding-0.6B',
            'model.encoder_dtype=bfloat16',
            'model.hidden_size=512',
            'model.n_heads=8',
            'model.n_blocks=6',
            'model.decoder_n_blocks=2',
            'strategy.find_unused_parameters=true',
            'loader.global_batch_size=16',
            'loader.eval_global_batch_size=16',
            f'trainer.max_steps={max_steps}',
            'trainer.precision=bf16-mixed',
        ]

    # Notebook policy: if CUDA is not available, run what we can on CPU.
    if not CUDA_AVAILABLE:
        model_specific.extend([
            'trainer.accelerator=cpu',
            'trainer.precision=32-true',
        ])

    if extra_overrides:
        model_specific.extend(extra_overrides)

    return common + model_specific


def find_checkpoint(run_dir: str) -> str:
    run_path = Path(run_dir)
    candidates = [
        run_path / 'checkpoints' / 'best.ckpt',
        run_path / 'checkpoints' / 'last.ckpt',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    attempted = '\n'.join(f'- {p.as_posix()}' for p in candidates)
    raise FileNotFoundError(f'No checkpoint found. Attempted:\n{attempted}')


def run_eval(model_key: str, run_dir: str, ckpt_path: str) -> dict:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = Path(run_dir) / 'eval' / 'gen_ppl' / timestamp
    out_dir.mkdir(parents=True, exist_ok=True)

    samples_path = out_dir / 'samples.pt'
    samples_txt_path = out_dir / 'samples.txt'
    metrics_path = out_dir / 'gen_ppl_metrics.json'

    gen_batch_size = EVAL_GEN_BATCH[model_key]
    num_steps = EVAL_NUM_STEPS[model_key]
    device = 'cuda' if CUDA_AVAILABLE else 'cpu'

    gen_cmd = [
        PYTHON_BIN,
        '-u',
        '-m',
        'discrete_diffusion.evaluations.generate_samples',
        f'checkpoint_path={ckpt_path}',
        f'samples_path={samples_path.as_posix()}',
        f'num_samples={NUM_SAMPLES}',
        f'batch_size={gen_batch_size}',
        f'num_steps={num_steps}',
        f'device={device}',
        'torch_compile=false',
        'save_text=true',
    ]

    try:
        run_cmd(gen_cmd)
    except subprocess.CalledProcessError:
        # Requested policy: continuous can drop generation batch size if OOM.
        if model_key == 'continuous' and gen_batch_size > 8:
            print('Continuous generation failed; retrying with batch_size=8')
            gen_cmd = [
                PYTHON_BIN,
                '-u',
                '-m',
                'discrete_diffusion.evaluations.generate_samples',
                f'checkpoint_path={ckpt_path}',
                f'samples_path={samples_path.as_posix()}',
                f'num_samples={NUM_SAMPLES}',
                'batch_size=8',
                f'num_steps={num_steps}',
                f'device={device}',
                'torch_compile=false',
                'save_text=true',
            ]
            run_cmd(gen_cmd)
        else:
            raise

    ppl_cmd = [
        PYTHON_BIN,
        '-u',
        '-m',
        'discrete_diffusion.evaluations.generative_ppl',
        f'samples_path={samples_path.as_posix()}',
        'model_tokenizer=gpt2',
        f'pretrained_model={EVAL_MODEL}',
        f'batch_size={EVAL_BATCH_SIZE}',
        f'max_length={MAX_LENGTH}',
        'retokenize=true',
        'first_chunk_only=true',
        f'metrics_path={metrics_path.as_posix()}',
        'torch_compile=false',
    ]
    run_cmd(ppl_cmd)

    with open(metrics_path, 'r', encoding='utf-8') as handle:
        metrics = json.load(handle)

    return {
        'metrics': metrics,
        'samples_path': samples_path.as_posix(),
        'samples_txt_path': samples_txt_path.as_posix(),
        'metrics_path': metrics_path.as_posix(),
        'eval_dir': out_dir.as_posix(),
    }


def load_samples_preview(samples_txt_path: str, n: int = 2) -> list[str]:
    path = Path(samples_txt_path)
    if not path.exists():
        return []

    raw = path.read_text(encoding='utf-8')
    sections = [section.strip() for section in raw.split('-' * 80) if section.strip()]
    return sections[:n]


def record_result(row: dict) -> pd.DataFrame:
    RESULT_ROWS.append(row)
    df = pd.DataFrame(RESULT_ROWS)
    SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(SUMMARY_CSV, index=False)
    return df


In [ ]:
def run_experiment(
    model_key: str,
    variant: str,
    extra_overrides: list[str] | None = None,
    max_steps: int | None = None,
) -> dict:
    if model_key not in MODEL_ENABLED:
        raise ValueError(f'Unknown model key: {model_key}')

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name = f'{model_key}_{variant}_{timestamp}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    VARIANT_OVERRIDES[(model_key, variant)] = extra_overrides or []

    if not MODEL_ENABLED[model_key]:
        train_status = 'skipped_disabled'
        eval_status = 'skipped_disabled'
        if model_key == 'continuous' and not CUDA_AVAILABLE:
            train_status = 'skipped_no_cuda'
            eval_status = 'skipped_no_cuda'

        row = {
            'model': model_key,
            'variant': variant,
            'run_name': run_name,
            'train_status': train_status,
            'eval_status': eval_status,
            'ppl': None,
            'avg_nll': None,
            'tokens': None,
            'train_wall_time_sec': 0.0,
            'eval_wall_time_sec': 0.0,
            'checkpoint_path': '',
            'metrics_path': '',
        }
        record_result(row)
        print(f'[{train_status}] {run_name}')
        return row

    train_cmd = [
        PYTHON_BIN,
        '-u',
        '-m',
        'discrete_diffusion',
        *train_overrides(
            model_key=model_key,
            run_dir=run_dir.as_posix(),
            run_name=run_name,
            max_steps=max_steps,
            extra_overrides=extra_overrides,
        ),
    ]

    train_start = time.time()
    train_status = 'ok'
    eval_status = 'not_run'
    checkpoint_path = ''
    metrics_path = ''
    ppl = None
    avg_nll = None
    tokens = None
    eval_elapsed = 0.0

    try:
        run_cmd(train_cmd)
    except subprocess.CalledProcessError as exc:
        train_status = f'failed_exit_{exc.returncode}'
        eval_status = 'skipped_train_failed'

    train_elapsed = time.time() - train_start

    if train_status == 'ok':
        try:
            checkpoint_path = find_checkpoint(run_dir.as_posix())
        except FileNotFoundError:
            train_status = 'failed_no_checkpoint'
            eval_status = 'skipped_train_failed'

    if train_status == 'ok' and checkpoint_path:
        eval_start = time.time()
        try:
            eval_outputs = run_eval(model_key, run_dir.as_posix(), checkpoint_path)
            eval_status = 'ok'
            metrics_path = eval_outputs['metrics_path']
            metrics = eval_outputs['metrics']
            ppl = metrics.get('ppl')
            avg_nll = metrics.get('avg_nll')
            tokens = metrics.get('tokens')

            preview = load_samples_preview(eval_outputs['samples_txt_path'], n=2)
            if preview:
                print('\nSample preview:')
                for sample in preview:
                    print('---')
                    print(sample[:800])
        except subprocess.CalledProcessError as exc:
            eval_status = f'failed_exit_{exc.returncode}'
        except Exception as exc:  # noqa: BLE001
            eval_status = f'failed_{type(exc).__name__}'
        eval_elapsed = time.time() - eval_start

    row = {
        'model': model_key,
        'variant': variant,
        'run_name': run_name,
        'train_status': train_status,
        'eval_status': eval_status,
        'ppl': ppl,
        'avg_nll': avg_nll,
        'tokens': tokens,
        'train_wall_time_sec': round(train_elapsed, 2),
        'eval_wall_time_sec': round(eval_elapsed, 2),
        'checkpoint_path': checkpoint_path,
        'metrics_path': metrics_path,
    }

    df = record_result(row)
    print('\nRecorded row:')
    display(df.tail(1))
    return row


BASELINE_GRID = {
    'mdlm': [
        ('baseline', ['optim.lr=3e-4'], FAST_STEP_BUDGETS['mdlm']),
    ],
    'latent_jepa': [
        ('baseline', ['algo.lambda_var=0.1', 'algo.lambda_cov=0.04'], FAST_STEP_BUDGETS['latent_jepa']),
    ],
    'continuous': [
        ('baseline', ['algo.parameterization=x0'], FAST_STEP_BUDGETS['continuous']),
    ],
}

ABLATION_GRID = {
    'mdlm': [
        ('ablation_baseline_lr_3e4', ['optim.lr=3e-4']),
        ('ablation_A_lr_1e4', ['optim.lr=1e-4']),
        ('ablation_B_lr_6e4', ['optim.lr=6e-4']),
    ],
    'latent_jepa': [
        ('ablation_baseline_v01_c004', ['algo.lambda_var=0.1', 'algo.lambda_cov=0.04']),
        ('ablation_A_v005_c002', ['algo.lambda_var=0.05', 'algo.lambda_cov=0.02']),
        ('ablation_B_v02_c008', ['algo.lambda_var=0.2', 'algo.lambda_cov=0.08']),
    ],
    'continuous': [
        ('ablation_baseline_x0', ['algo.parameterization=x0']),
        ('ablation_A_epsilon', ['algo.parameterization=epsilon']),
        ('ablation_B_small_arch', ['model.hidden_size=384', 'model.n_heads=6', 'model.n_blocks=4']),
    ],
}

RUN_SMOKE_MDLM = False
RUN_BASELINES = True
RUN_ABLATIONS = True

if RUN_SMOKE_MDLM:
    _ = run_experiment(
        model_key='mdlm',
        variant='smoke_baseline',
        extra_overrides=['optim.lr=3e-4'],
        max_steps=200,
    )

if RUN_BASELINES:
    for model_key, entries in BASELINE_GRID.items():
        for variant, extra_overrides, steps in entries:
            _ = run_experiment(
                model_key=model_key,
                variant=variant,
                extra_overrides=extra_overrides,
                max_steps=steps,
            )

if RUN_ABLATIONS:
    for model_key, entries in ABLATION_GRID.items():
        for variant, extra_overrides in entries:
            _ = run_experiment(
                model_key=model_key,
                variant=variant,
                extra_overrides=extra_overrides,
                max_steps=ABLATION_MAX_STEPS,
            )

print(f'\nSummary CSV: {SUMMARY_CSV.as_posix()}')


In [ ]:
if SUMMARY_CSV.exists():
    summary_df = pd.read_csv(SUMMARY_CSV)
else:
    summary_df = pd.DataFrame(RESULT_ROWS)

if summary_df.empty:
    print('No runs recorded yet.')
else:
    display(summary_df)

    valid_df = summary_df[
        (summary_df['train_status'] == 'ok')
        & (summary_df['eval_status'] == 'ok')
        & (summary_df['ppl'].notna())
    ].copy()

    if valid_df.empty:
        print('No successful eval rows yet.')
    else:
        leaderboard = valid_df.sort_values(['model', 'ppl'], ascending=[True, True])
        print('Leaderboard (lower PPL is better):')
        display(leaderboard[['model', 'variant', 'run_name', 'ppl', 'avg_nll', 'tokens']])


In [ ]:
if 'valid_df' not in globals() or valid_df.empty:
    print('No successful results to plot yet.')
else:
    sns.set_theme(style='whitegrid')

    g = sns.catplot(
        data=valid_df,
        kind='bar',
        x='variant',
        y='ppl',
        col='model',
        sharey=False,
        height=4,
        aspect=1.2,
    )
    g.set_xticklabels(rotation=45, horizontalalignment='right')
    g.fig.subplots_adjust(top=0.82)
    g.fig.suptitle('Tiny Shakespeare Gen-PPL by Model and Variant')
    plt.show()

    plt.figure(figsize=(7, 5))
    sns.scatterplot(
        data=valid_df,
        x='train_wall_time_sec',
        y='ppl',
        hue='model',
        style='variant',
        s=100,
    )
    plt.title('Train Wall Time vs Gen-PPL')
    plt.xlabel('train_wall_time_sec')
    plt.ylabel('ppl')
    plt.tight_layout()
    plt.show()


In [ ]:
def suggest_next_steps(df: pd.DataFrame) -> list[str]:
    if df.empty:
        return ['No successful eval rows yet.']

    suggestions: list[str] = []

    for model_key in sorted(df['model'].unique()):
        mdf = df[df['model'] == model_key].copy()
        if mdf.empty:
            continue

        mdf = mdf.sort_values('ppl')
        best = mdf.iloc[0]
        suggestions.append(
            f"{model_key}: best variant is '{best['variant']}' (ppl={best['ppl']:.4f}, run={best['run_name']})."
        )

        baseline_rows = mdf[mdf['variant'].str.contains('baseline', case=False, na=False)]
        if baseline_rows.empty:
            follow_candidates = mdf.head(2)
            for _, row in follow_candidates.iterrows():
                ovs = VARIANT_OVERRIDES.get((model_key, row['variant']), [])
                ov_text = ', '.join(ovs) if ovs else '<none>'
                suggestions.append(
                    f"{model_key}: follow up around '{row['variant']}' with small neighborhood search around [{ov_text}]."
                )
            continue

        baseline = baseline_rows.iloc[0]
        base_ppl = float(baseline['ppl'])

        candidates = []
        for _, row in mdf.iterrows():
            if row['variant'] == baseline['variant']:
                continue
            delta = base_ppl - float(row['ppl'])
            candidates.append((delta, row))

        candidates.sort(key=lambda item: item[0], reverse=True)
        top_two = candidates[:2]

        if not top_two:
            suggestions.append(f"{model_key}: no non-baseline variants to compare yet.")
            continue

        for delta, row in top_two:
            ovs = VARIANT_OVERRIDES.get((model_key, row['variant']), [])
            ov_text = ', '.join(ovs) if ovs else '<none>'
            suggestions.append(
                f"{model_key}: candidate '{row['variant']}' delta_vs_baseline={delta:.4f}. Follow-up: narrow search around [{ov_text}]."
            )

    return suggestions


if 'valid_df' not in globals() or valid_df.empty:
    print('No successful eval rows to generate suggestions.')
else:
    next_steps = suggest_next_steps(valid_df)
    print('Improvement candidates:')
    for i, suggestion in enumerate(next_steps, start=1):
        print(f'{i}. {suggestion}')
